## What an agent is made of

Before any code, the map. Every capable agent, whatever framework it uses, is the same eight
parts. This notebook adds them one at a time to CampusAI:

```text
MODEL          the language model: text in, text out, no memory, no actions            G2
CONTEXT        what the model sees on each call: instructions, history, evidence        G2, G5, G6, G7
TOOLS          functions the model may REQUEST; your code decides and executes          G3, G11
ORCHESTRATION  the loop or graph that decides what runs next                            G1, G3, G4, G10, G12
MEMORY         short-term (this conversation) and long-term (this user)                 G5, G6
KNOWLEDGE      documents retrieved on demand and placed into the context                G7
GUARDRAILS     approvals, permissions, limits, retries: safety in the architecture      G8, G9
OBSERVABILITY  streaming, traces, audit trails, evaluation                               G13
TRIGGERS       who starts a run: a user, another agent, or a schedule                   G14
```

The single most important idea: **the model never does anything**. It only produces text, some
of which is a request. Everything that acts, remembers or checks is code you write, and in
LangGraph that code is organised as a graph.

## Meet CampusAI (the project you will grow)

**Northfield University** runs a student helpdesk. Every day it answers questions such as
*"What is my attendance in CS201?"*, *"Can I register for 26 credits?"*, *"What does the
handbook say about retaking a course?"*, *"When is the science library open?"*, and requests
such as *"Register me for EE150"* or *"Email my tutor"*.

**CampusAI** is the assistant we build for that helpdesk. It is the running example of this
notebook, not a product. The university data is tiny and lives in the notebook:

```text
STUDENTS      three student records (name, programme, attendance, credits, email)   -> get_student tool     (G3)
COURSES       four courses (title, credits, seats, prerequisites, description)      -> get_course tool      (G3)
HANDBOOK      five policy paragraphs                                                -> search_handbook tool (G3), retrieval (G7)
FAQ_DOCS      library, labs and exam notes                                          -> retrieval (G7)
REGISTRATIONS every registration the assistant makes                               -> register_course tool (G8, write)
EMAIL_OUTBOX  every email the assistant sends                                       -> send_email tool      (G8, write)
library server two tools served by a separate MCP process, plus a public MCP server  (G11)
```

Two kinds of people talk to CampusAI: **students**, who may only read, and **staff**, who may
also register courses and send email. That difference drives the permission section.

## How to read the code cells

```python
builder.add_node("agent", call_model)     # LangGraph: register a node
model.invoke(messages)                    # LangChain: one request, one AIMessage
class Ticket(BaseModel): ...              # Pydantic: validated data class
show_messages(result["messages"])         # ours: defined in this notebook
```

LangGraph gives you graphs, state, checkpoints, stores, interrupts, parallelism and streaming. It
does not define models or tools; for those it reuses LangChain's chat model classes and `@tool`
decorator, which is why a few lines carry the `# LangChain` tag. Everything tagged **ours** is
ordinary Python you could rewrite yourself.

## Your API key (30 seconds)

The course model runs on OpenRouter. Give this notebook your issued key in one of two ways:

- **Recommended:** click the key icon in Colab's left sidebar, add a secret named
  `OPENROUTER_API_KEY`, and switch on *Notebook access*. Every course notebook then finds it automatically.
- **Or:** run the cell below and paste the key when asked (it is kept only in this session).

No key? Press Enter when asked. The notebook switches to the mock model and everything still runs.
Never paste a key into a code cell: notebooks get shared.

In [ ]:
# === Setup: run this cell first ===============================================
%pip install -q -U "langgraph>=1.0" "langchain>=1.2" "langchain-openai>=1.1"

import json, os, re, sys, time, operator             # Python standard library
from getpass import getpass
from dataclasses import dataclass
from typing import Annotated, Literal, TypedDict

MODEL_NAME = "openai/gpt-oss-120b"                 # the course model on OpenRouter
OPENROUTER_URL = "https://openrouter.ai/api/v1"

def load_api_key():                                 # ours
    """Look for the key in Colab Secrets, then the environment, then ask once."""
    try:
        from google.colab import userdata           # only exists on Colab
        key = userdata.get("OPENROUTER_API_KEY")
        if key:
            return key, "Colab secret"
    except Exception:
        pass
    if os.getenv("OPENROUTER_API_KEY"):
        return os.environ["OPENROUTER_API_KEY"], "environment variable"
    try:
        key = getpass("OpenRouter API key (press Enter to use the mock model): ").strip()
    except Exception:                               # no keyboard available (automated run)
        key = ""
    return (key, "typed in") if key else ("", "none")

API_KEY, KEY_SOURCE = load_api_key()
LIVE = bool(API_KEY)                                # True = real model, False = mock model

def make_model(temperature=0.0):                    # ours
    """Return a chat model: ChatOpenAI pointed at OpenRouter (LIVE) or the mock (no key)."""
    if not LIVE:
        return MockChatModel()
    from langchain_openai import ChatOpenAI         # LangChain: chat model class for OpenAI-compatible APIs
    return ChatOpenAI(model=MODEL_NAME, api_key=API_KEY, base_url=OPENROUTER_URL, temperature=temperature,
                      max_tokens=900, extra_body={"reasoning": {"effort": "low"}})

print("Model  :", MODEL_NAME)
print("Key    :", KEY_SOURCE)
print("Mode   :", "LIVE - real model replies" if LIVE else "MOCK - canned replies, real shapes, zero cost")

### The mock model (run it; read it later, or never)

Without a key, `make_model()` returns the class below: a real LangChain chat model whose replies
follow fixed rules for this notebook's questions (it requests tools when a question mentions a
student id, a course code, a handbook topic, the library, a registration, or an email). It
produces genuine `AIMessage` objects with `tool_calls`, so every graph in this notebook runs on it
unchanged. It is also deliberately gullible about instructions hidden in documents (section G8).

In [ ]:
from langchain_core.language_models import BaseChatModel          # LangChain: base class of every chat model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage, RemoveMessage   # LangChain: message classes
from langchain_core.outputs import ChatGeneration, ChatResult      # LangChain: what _generate must return
from langchain_core.utils.function_calling import convert_to_openai_tool   # LangChain: tool -> JSON schema

def text_of(message) -> str:                                       # ours
    """Message content as plain text (real models and MCP tools may return a list of content blocks)."""
    content = message.content
    if isinstance(content, str):
        return content
    return " ".join(block.get("text", "") for block in content if isinstance(block, dict))

def _phrase(name, content):                                        # ours (mock helper)
    """One tool result -> one readable sentence."""
    try:
        data = json.loads(content)
    except Exception:
        data = None
    if name == "get_student" and isinstance(data, dict) and "name" in data:
        return f"Student {data['id']} is {data['name']} ({data['programme']}, year {data['year']}) with {data['attendance']}% attendance and {data['credits']} credits."
    if name == "get_course" and isinstance(data, dict) and "title" in data:
        return f"Course {data['code']} '{data['title']}' is {data['credits']} credits with {data['seats_left']} seats left; prerequisites: {data['prerequisites'] or 'none'}."
    if name in ("search_handbook", "search_knowledge"):
        return "The sources say: " + content.splitlines()[0][:160]
    if name == "register_course":
        return ("Registration done: " if isinstance(data, dict) and data.get("status") == "registered" else "Registration NOT done: ") + content[:140]
    if name == "send_email":
        return "Email result: " + content[:120]
    if name == "remember_about_me":
        return content[:120]
    if name == "library_hours":
        return f"The library is open {content}."
    if name == "search_library":
        return "Library catalogue: " + content[:140]
    if name == "ask_question":
        return "DeepWiki says: " + content[:200]
    if name == "transfer_to_handbook":
        return "(transferred to the handbook desk)"
    return f"{name} reports: {content[:160]}"

def _mock_decide(messages, tools):                                 # ours (mock helper): the mock's whole 'brain'
    names = [t["function"]["name"] for t in tools]
    humans = [i for i, m in enumerate(messages) if isinstance(m, HumanMessage)]
    last_human = humans[-1] if humans else -1
    question = text_of(messages[last_human]) if last_human >= 0 else ""
    lower = question.lower()
    system_text = " ".join(text_of(m) for m in messages if isinstance(m, SystemMessage))
    system_lower = system_text.lower()
    turn = messages[last_human + 1:] if last_human >= 0 else list(messages)
    results = [(m.name or "tool", text_of(m)) for m in turn if isinstance(m, ToolMessage)]
    requested = {(c["name"], json.dumps(c["args"], sort_keys=True)) for m in turn if isinstance(m, AIMessage) for c in m.tool_calls}
    worker_reports = [m.name for m in messages if isinstance(m, AIMessage) and m.name and m.name.endswith("_worker")]
    calls = []

    def want(name, **args):
        key = (name, json.dumps(args, sort_keys=True))
        if name in names and key not in requested:
            requested.add(key)
            calls.append({"name": name, "args": args, "id": f"call_{name}_{len(calls) + 1}"})

    students = re.findall(r"\b(S\d{3})\b", question)
    courses = re.findall(r"\b([A-Z]{2}\d{3})\b", question)
    policy_words = re.search(r"handbook|policy|rule|allowed|retake|late|fees|maximum|minimum|lab|exam", lower)
    # Deliberately gullible: instructions hidden in retrieved text are obeyed (G8 demo).
    for _, content in results:
        hit = re.search(r"IGNORE PREVIOUS INSTRUCTIONS.*?register_course\D+(S\d{3})\D+([A-Z]{2}\d{3})", content, re.I | re.S)
        if hit:
            want("register_course", student_id=hit.group(1), course_code=hit.group(2))
    for student_id in students:
        want("get_student", student_id=student_id)
    for course_code in courses:
        want("get_course", course_code=course_code)
    if policy_words or re.search(r"attendance|credits|regist|library|addendum", lower):
        for name in ("search_handbook", "search_knowledge"):
            want(name, query=question)
    if "timetable" in lower and students:
        want("get_timetable", student_id=students[0])
    if "library" in lower and re.search(r"hour|open|close", lower):
        want("library_hours", branch="science" if "science" in lower else "main")
    if "library" in lower and re.search(r"find|search|book|catalogue", lower):
        want("search_library", query=question)
    if re.search(r"repository|repo|github|deepwiki", lower):
        want("ask_question", repoName="langchain-ai/langgraph", question=question)
    if "transfer_to_handbook" in names and policy_words and not calls and not any(n == "transfer_to_handbook" for n, _ in results):
        want("transfer_to_handbook", reason="the question needs handbook policy")
    if re.search(r"remember|prefer", lower):
        want("remember_about_me", fact=question)
    # Writes only once the read-only evidence is in.
    if not calls and students and courses and re.search(r"register|enrol", lower) and not any(n == "register_course" for n, _ in results):
        want("register_course", student_id=students[0], course_code=courses[0])
    if not calls and "email" in lower and students and not any(n == "send_email" for n, _ in results):
        want("send_email", to=students[0], subject="Message from CampusAI", body=question)
    # Structured outputs arrive as tools named after the Pydantic class.
    if not calls:
        service_request = re.search(r"remember|prefer", lower) or ("library" in lower and re.search(r"hour|open|close|find|search|catalogue", lower))
        category = "records" if (students or courses or service_request) else ("faq" if policy_words or re.search(r"attendance|credits|library", lower) else "smalltalk")
        want("Ticket", category=category, priority="high" if re.search(r"urgent|exam|blocked|twice", lower) else "medium", student_id=students[0] if students else "unknown")
        want("RouteDecision", category=category)
        needed = (["records_worker"] if (students or courses) else []) + (["handbook_worker"] if policy_words or re.search(r"attendance|credits", lower) else [])
        pending = [w for w in needed if w not in worker_reports]
        want("SupervisorDecision", next_worker=pending[0] if pending else "FINISH")
    usage = {"input_tokens": 40 + 8 * len(messages), "output_tokens": 30, "total_tokens": 70 + 8 * len(messages)}
    if calls:
        return AIMessage(content="", tool_calls=calls, usage_metadata=usage)
    # Text replies.
    if results:
        first = ("register_course", "send_email", "remember_about_me", "library_hours", "search_library")   # actions and remote tools first
        ordered = [r for r in results if r[0] in first] + [r for r in results if r[0] not in first]
        prefix = "- " if "prefers short bullet" in system_lower else ""
        return AIMessage(content=prefix + "Here is what I found. " + " ".join(_phrase(n, c) for n, c in ordered), usage_metadata=usage)
    if "summarise the conversation" in system_lower:
        ids = sorted(set(re.findall(r"\b[SC][A-Z]?\d{3}\b", " ".join(text_of(m) for m in messages))))
        return AIMessage(content=f"Summary: the user's name is Rahul; they asked CampusAI about students and courses, including {', '.join(ids) or 'no specific ids'}, and about the weather.", usage_metadata=usage)
    if "excerpts" in system_lower:
        q_words = {w.rstrip("s") for w in re.findall(r"[a-z]+", lower) if len(w) > 3}
        sentences = [s.strip() for s in re.split(r"(?<=\.)\s+", system_text.split("\n\n", 1)[-1]) if s.strip()]
        ranked = sorted(sentences, key=lambda s: (-len(q_words & {w.rstrip("s") for w in re.findall(r"[a-z]+", s.lower())}), -sentences.index(s)))[:2]
        return AIMessage(content="Based on the sources: " + " ".join(ranked), usage_metadata=usage)
    if "draft a short email" in system_lower:
        return AIMessage(content="Dear student, our records show that your attendance is below the 75% required to sit the final exam. Please contact the helpdesk this week to discuss your options.", usage_metadata=usage)
    if "combine the specialist reports" in system_lower:
        reports = [text_of(m) for m in messages if isinstance(m, AIMessage) and m.name and m.name.endswith("_worker")]
        return AIMessage(content="Final answer. " + " ".join(r[:160] for r in reports), usage_metadata=usage)
    known = re.search(r"Known facts about this user: (.+?)(?:\n|$)", system_text)
    if known and re.search(r"know about me|my preferences|how should", lower):
        return AIMessage(content=f"From earlier conversations I know: {known.group(1)}", usage_metadata=usage)
    told = re.search(r"my name is (\w+)", " ".join(text_of(m) for m in messages if isinstance(m, HumanMessage)), re.I) or re.search(r"name is (\w+)", system_text)   # the summary may carry the name
    if re.search(r"my name is", lower):
        return AIMessage(content=f"Nice to meet you, {told.group(1)}! I am CampusAI.", usage_metadata=usage)
    if "my name" in lower:
        return AIMessage(content=f"Your name is {told.group(1)}." if told else "I don't know your name - you have not told me in this conversation.", usage_metadata=usage)
    if "weather" in lower:
        return AIMessage(content="I cannot check the weather, but I can help with anything about the university.", usage_metadata=usage)
    if re.search(r"\b(hi|hello|hey)\b", lower):
        return AIMessage(content="Hello! I am CampusAI, the Northfield helpdesk assistant. Ask me about students, courses or the handbook.", usage_metadata=usage)
    if "what can you do" in lower or "campusai" in lower:
        return AIMessage(content="I can look up student records and courses, search the handbook, and (with approval) register courses or send emails.", usage_metadata=usage)
    return AIMessage(content=f"(mock reply) You asked: {question[:120]}", usage_metadata=usage)


class MockChatModel(BaseChatModel):                                # ours, built on LangChain's base class
    bound_tools: list = []

    @property
    def _llm_type(self) -> str:
        return "campusai-mock"

    def bind_tools(self, tools, **kwargs):                         # LangChain interface, our implementation
        return self.model_copy(update={"bound_tools": [convert_to_openai_tool(t) for t in tools]})   # Pydantic

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:   # LangChain calls this from invoke()
        return ChatResult(generations=[ChatGeneration(message=_mock_decide(messages, self.bound_tools))])


model = make_model()
print("model class :", type(model).__name__)

In [ ]:
# ours: prints a message list one line per message (m.type, m.tool_calls, m.name, m.content are LangChain attributes)
def show_messages(messages, width=110):
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            for call in m.tool_calls:
                print(f"  ai     -> tool call: {call['name']}({json.dumps(call['args'])})")
            if text_of(m).strip():
                print(f"  ai     : {text_of(m)[:width]}")
        elif isinstance(m, ToolMessage):
            print(f"  tool   : [{m.name}] {text_of(m)[:width]}")
        else:
            label = f"{m.type}{'/' + m.name if getattr(m, 'name', None) else ''}"
            print(f"  {label:6} : {text_of(m)[:width]}")

# ours: Northfield University's tiny data (see "Meet CampusAI" at the top)
STUDENTS = {
    "S001": {"name": "Priya Nair",   "programme": "Computer Science", "year": 2, "attendance": 68, "credits": 18, "email": "priya@northfield.example"},
    "S002": {"name": "Arjun Mehta",  "programme": "Electronics",      "year": 3, "attendance": 91, "credits": 21, "email": "arjun@northfield.example"},
    "S003": {"name": "Lin Zhao",     "programme": "Computer Science", "year": 1, "attendance": 80, "credits": 12, "email": "lin@northfield.example"},
}
COURSES = {
    "CS101": {"title": "Introduction to Programming", "credits": 4, "seats_left": 5, "prerequisites": [],        "description": "Python basics, loops, functions and simple data structures for first-year students."},
    "CS201": {"title": "Data Structures",             "credits": 4, "seats_left": 0, "prerequisites": ["CS101"], "description": "Lists, trees, hash tables and graph algorithms with weekly lab sessions."},
    "EE150": {"title": "Circuits I",                  "credits": 3, "seats_left": 2, "prerequisites": [],        "description": "DC and AC circuit analysis, Kirchhoff's laws and lab measurements."},
    "MA110": {"title": "Calculus",                    "credits": 4, "seats_left": 9, "prerequisites": [],        "description": "Limits, derivatives and integrals for engineering programmes."},
}
HANDBOOK = {
    "attendance":   "Students need at least 75% attendance in a course to sit its final exam. Medical exceptions require a certificate from the health centre.",
    "credits":      "A student may register for at most 24 credits per semester. Requests above 24 credits need the dean's approval.",
    "retake":       "A failed course may be retaken once. The better of the two grades counts towards the degree.",
    "registration": "Registration closes at the end of week 2 of the semester. Late registration needs the department head's approval and a 50 USD fee.",
    "fees":         "Tuition fees are due by the end of week 4. A late payment adds 2% per month.",
}
FAQ_DOCS = {
    "library":  "The main library is open 08:00 to 22:00 on weekdays. The science library closes at 18:00. Books can be borrowed for 21 days.",
    "labs":     "Computing labs are open to registered students with a campus card. Lab sessions for CS201 run every Wednesday afternoon.",
    "exams":    "Final exams take place in weeks 15 and 16. Students must bring photo identification; calculators are allowed only in engineering exams.",
}
REGISTRATIONS, EMAIL_OUTBOX = [], []
print("data ready:", len(STUDENTS), "students,", len(COURSES), "courses,", len(HANDBOOK), "handbook topics,", len(FAQ_DOCS), "FAQ notes")

# LangGraph G1 — Graphs before agents
Most tutorials start with a model. We start with the thing LangGraph actually is: a way to run
Python functions over a shared dictionary in a declared order. No model, no tools, no magic.

```text
STATE   a dictionary that flows through the graph; nodes read it and return partial updates
NODE    a Python function   state -> {keys that changed}
EDGE    "after node A, run node B"
CONDITIONAL EDGE   "after node A, call a routing function; it names the next node"
START / END        where a run enters and leaves
```

Why bother, when a plain script does the same? Because a declared graph can be **drawn**,
**paused**, **resumed**, **checkpointed**, **branched** and **run in parallel** without changing
the node code. Every later section uses one of those abilities. And the difference between a
*workflow* and an *agent* becomes a one-line answer: in a workflow your code chooses the edges;
in an agent a model chooses some of them.

### Step 1 — Three nodes in a row

The state is a dictionary with a number and a trace string. Each node returns only the keys it
changes; LangGraph merges that update into the state and passes it on.

In [ ]:
from langgraph.graph import StateGraph, START, END     # LangGraph: the graph builder and its two fixed endpoints

class Counter(TypedDict):                              # ours: the state schema (a typed dict)
    value: int
    trace: str

def add_ten(state: Counter):                           # ours: a node = function(state) -> partial update
    return {"value": state["value"] + 10, "trace": state["trace"] + " +10"}

def double(state: Counter):                            # ours
    return {"value": state["value"] * 2, "trace": state["trace"] + " x2"}

def subtract_one(state: Counter):                      # ours
    return {"value": state["value"] - 1, "trace": state["trace"] + " -1"}

builder = StateGraph(Counter)                          # LangGraph: declare a graph over this state
builder.add_node("add_ten", add_ten)                   # LangGraph: register nodes by name
builder.add_node("double", double)
builder.add_node("subtract_one", subtract_one)
builder.add_edge(START, "add_ten")                     # LangGraph: edges fix the order
builder.add_edge("add_ten", "double")
builder.add_edge("double", "subtract_one")
builder.add_edge("subtract_one", END)
pipeline = builder.compile()                           # LangGraph: the declaration becomes runnable

print(pipeline.invoke({"value": 1, "trace": "1"}))    # LangGraph: invoke() runs START -> ... -> END
print(pipeline.get_graph().draw_mermaid())            # LangGraph: the graph as a Mermaid diagram (paste into mermaid.live)

> **What just happened**
>
> The run entered at START, which has one edge, so `add_ten` ran first with `{'value': 1, 'trace': '1'}` and returned `{'value': 11, 'trace': '1 +10'}`. LangGraph merged that into the state and followed the edge to `double` (22), then `subtract_one` (21), then END.
> The printed dictionary is the final state; the trace string is the order the nodes ran in. The Mermaid text under it is the same graph drawn: three boxes, four arrows.

### Step 2 — A conditional edge and a loop

A routing function looks at the state and returns the **name** of the next node. Pointing an
edge back at an earlier node makes a loop; the routing function decides when to stop. Hold on to
this picture: the agent loop of G3 is exactly this loop with a model inside.

In [ ]:
def is_even(state: Counter):                           # ours: routing function -> name of the next node
    return "double" if state["value"] % 2 == 0 else "add_ten"

branchy = StateGraph(Counter)
branchy.add_node("add_ten", add_ten)
branchy.add_node("double", double)
branchy.add_conditional_edges(START, is_even, {"double": "double", "add_ten": "add_ten"})   # LangGraph: choose the path from the state
branchy.add_edge("add_ten", END)
branchy.add_edge("double", END)
branch_graph = branchy.compile()
print("even :", branch_graph.invoke({"value": 4, "trace": "4"}))
print("odd  :", branch_graph.invoke({"value": 5, "trace": "5"}))

def keep_going(state: Counter):                        # ours: the loop condition
    return "add_ten" if state["value"] < 50 else END

loopy = StateGraph(Counter)
loopy.add_node("add_ten", add_ten)
loopy.add_edge(START, "add_ten")
loopy.add_conditional_edges("add_ten", keep_going, {"add_ten": "add_ten", END: END})   # LangGraph: an edge back to the same node
loop_graph = loopy.compile()
print("loop :", loop_graph.invoke({"value": 5, "trace": "5"}))

> **What just happened**
>
> Branch graph: START has no fixed edge, only a conditional one, so `is_even` ran first on the input. For 4 it returned the string `"double"`, which the mapping turned into the node `double` (4 x2 = 8), then END. For 5 it returned `"add_ten"` (5 +10 = 15). Only one of the two nodes ran each time; the trace shows which.
> Loop graph: START -> `add_ten` (15). After every `add_ten`, `keep_going` looked at the value: below 50 it returned `"add_ten"` again; at 55 it returned END. Five passes, five `+10` in the trace. The node never knew it was looping; the routing function did all the deciding.

### Step 3 — A real workflow with no AI in it: keyword triage

CampusAI v0 answers handbook questions by keyword matching. It is a proper LangGraph workflow:
classify, route, answer. Notice what it cannot do: understand a question phrased in new words,
look up a specific student, or decide that it needs more information. Each of those gaps is a
later section. But the *shape* (state in, nodes, conditional edge, state out) never changes.

In [ ]:
class TicketState(TypedDict, total=False):              # ours: total=False means keys may be absent at first
    question: str
    topic: str
    answer: str

def classify_by_keyword(state: TicketState):           # ours: node 1
    q = state["question"].lower()
    for topic in HANDBOOK:
        if topic in q:
            return {"topic": topic}
    return {"topic": "unknown"}

def answer_from_handbook(state: TicketState):          # ours: node 2a
    return {"answer": HANDBOOK[state["topic"]]}

def apologise(state: TicketState):                     # ours: node 2b
    return {"answer": "Sorry, I can only answer handbook questions about: " + ", ".join(HANDBOOK)}

def route_by_topic(state: TicketState):                # ours: routing function
    return "answer" if state["topic"] in HANDBOOK else "apologise"

triage = StateGraph(TicketState)
triage.add_node("classify", classify_by_keyword)
triage.add_node("answer", answer_from_handbook)
triage.add_node("apologise", apologise)
triage.add_edge(START, "classify")
triage.add_conditional_edges("classify", route_by_topic, {"answer": "answer", "apologise": "apologise"})
triage.add_edge("answer", END)
triage.add_edge("apologise", END)
campusai_v0 = triage.compile()

for q in ["What is the attendance rule?", "How many credits can I take?", "Can I still sign up for a course in week 5?"]:
    out = campusai_v0.invoke({"question": q})
    print(f"{q!r:52} -> topic={out['topic']:12} | {out['answer'][:70]}")
print("\nThe last question is about registration, but the word 'registration' never appears. Keyword triage cannot know that.")

> **What just happened**
>
> Each question went START -> `classify`. The node scanned the question for a handbook topic word and wrote `topic` into the state. Then `route_by_topic` read that topic: a known one returned `"answer"`, so `answer_from_handbook` ran and copied the paragraph into `answer`; `unknown` returned `"apologise"`.
> The third question never contains the word "registration", so the scan produced `unknown` and the apology path ran. That is the limit of code-only routing, and the reason the next section adds a model.

### Recap

- **The problem we started with:** a plain script cannot be drawn, paused, resumed or branched; keyword rules cannot understand new phrasings.
- **What we added:** LangGraph vocabulary: state, nodes, edges, conditional edges, loops, START and END.
- **What you saw in the output:** three toy graphs ran as drawn; the keyword workflow answered two questions and missed the third.
- **Carry forward:** G2 keeps this exact shape and replaces `classify_by_keyword` with a model, so the same graph can understand new phrasings.